# Phase 33: Transformer Encoder Architecture

**Goal:** We are building the most advanced model in our benchmark: The **Transformer Encoder**. 
Unlike the LSTM which reads events sequentially (and suffers from forgetting), the Transformer uses **Multi-Head Self-Attention** to look at all 10 events simultaneously. 

We will implement the Pre-LN (Pre-Layer Norm) architecture from scratch, and crucially, we will mathematically extract the Attention Rollout (Abnar & Zuidema, 2020) so we can see *exactly* what the AI is paying attention to!

In [1]:
import os
import sys
!{sys.executable} -m pip install torch pytorch-lightning numpy pytest  # type: ignore  # pylint: disable=import-error

import warnings
warnings.filterwarnings("ignore")

import torch
import torch.nn as nn
import torch.nn.functional as F
import pytorch_lightning as pl
import numpy as np


### Step 1: Multi-Head Self-Attention Block (Subphase 33.1)
We mathematically implement $Attention(Q, K, V) = softmax(\frac{QK^T}{\sqrt{d_k}})V$. 
Crucially, we store `self.last_attention_weights` during every forward pass so we can extract it later for our XAI Research!

In [2]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError(f"d_model ({d_model}) must be divisible by n_heads ({n_heads})")
            
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads
        
        self.W_q = nn.Linear(d_model, d_model)
        self.W_k = nn.Linear(d_model, d_model)
        self.W_v = nn.Linear(d_model, d_model)
        self.W_o = nn.Linear(d_model, d_model)
        
        self.dropout = nn.Dropout(dropout)
        self.last_attention_weights = None
        
    def forward(self, x):
        batch_size, seq_len, _ = x.size()
        
        # Linear Projections & Reshape for multi-head
        Q = self.W_q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_k(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_v(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        
        # Scaled Dot-Product Attention: QK^T / sqrt(d_k)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        attention_weights = F.softmax(scores, dim=-1)
        
        # STORE the attention weights for XAI (detached so it doesn't break backprop memory)
        self.last_attention_weights = attention_weights.detach()
        
        # Apply dropout to attention and multiply by V
        attention_applied = torch.matmul(self.dropout(attention_weights), V)
        
        # Reshape back to original dimensions
        attention_applied = attention_applied.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        
        return self.W_o(attention_applied)

print("✅ Multi-Head Self-Attention mathematically constructed!")

✅ Multi-Head Self-Attention mathematically constructed!


### Step 2: Transformer Encoder Block & Network (Subphase 33.2)
We construct the Pre-LN Transformer. By placing the `LayerNorm` BEFORE the attention and feed-forward networks, we prevent the gradients from vanishing, resulting in highly stable deep learning!

In [3]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        # PRE-Layer Norms
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        
        self.attn = MultiHeadSelfAttention(d_model, n_heads, dropout)
        
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_ff, d_model)
        )
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x):
        # Pre-LN Architecture with Residual Connections
        x = x + self.dropout(self.attn(self.norm1(x)))
        x = x + self.dropout(self.ff(self.norm2(x)))
        return x

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))
        
    def forward(self, x):
        return x + self.pe[:, :x.size(1)]

class TransformerModel(pl.LightningModule):
    def __init__(self, n_features, n_classes, d_model=64, n_heads=4, n_layers=2, d_ff=128, dropout=0.1, max_seq_len=10):
        super().__init__()
        self.save_hyperparameters()
        
        self.input_proj = nn.Linear(n_features, d_model)
        self.pos_encoder = PositionalEncoding(d_model, max_len=max_seq_len)
        
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_layers)
        ])
        
        self.final_norm = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, n_classes)
        self.criterion = nn.CrossEntropyLoss()
        
    def forward(self, x):
        x = self.input_proj(x)
        x = self.pos_encoder(x)
        
        for layer in self.layers:
            x = layer(x)
            
        x = self.final_norm(x)
        
        # Mean Pooling over the sequence length to get the final representation
        x_pooled = x.mean(dim=1)
        return self.head(x_pooled)

print("✅ Pre-LN Transformer Architecture Generated!")

✅ Pre-LN Transformer Architecture Generated!


### Step 3: Attention Rollout XAI Extractor (Subphase 33.3)
To understand *why* the Transformer flagged a sequence as Malicious, we implement Attention Rollout. This mathematically fuses the attention matrices across all layers to output a single (batch, seq_len) array indicating which exact timestamp contained the attack signature!

In [4]:
class AttentionRolloutExtractor:
    def __init__(self):
        pass
        
    def extract(self, model: TransformerModel, X_seq: torch.Tensor) -> np.ndarray:
        model.eval()
        with torch.no_grad():
            # Run forward pass to populate last_attention_weights
            _ = model(X_seq)
            
            batch_size, seq_len, _ = X_seq.shape
            
            # Rollout Identity Matrix
            rollout = torch.eye(seq_len).unsqueeze(0).repeat(batch_size, 1, 1).to(X_seq.device)
            
            for layer in model.layers:
                # Get attention weights from this layer: (batch, n_heads, seq, seq)
                attn_weights = layer.attn.last_attention_weights
                # Average across all heads
                attn_weights_mean = attn_weights.mean(dim=1)
                
                # Add Identity to simulate residual connection: A = 0.5 * A + 0.5 * I
                attn_weights_mean = 0.5 * attn_weights_mean + 0.5 * torch.eye(seq_len).unsqueeze(0).to(X_seq.device)
                
                # Matrix multiply recursively to rollout the attention through the network
                rollout = torch.matmul(attn_weights_mean, rollout)
                
            # We extract the mean attention paid to each token across the entire sequence
            # Returning shape: (batch, seq_len)
            final_scores = rollout.mean(dim=1)
            
            # Normalize so they sum to 1.0
            final_scores = final_scores / final_scores.sum(dim=1, keepdim=True)
            return final_scores.cpu().numpy()

print("✅ Attention Rollout Mathematical Extractor Ready!")

✅ Attention Rollout Mathematical Extractor Ready!


### Step 4: Transformer Unit Tests (Subphase 33.4)
Transformers are incredibly complex and easy to write incorrectly. We run 4 mathematical unit tests before committing GPU resources!

In [5]:
print("=== RUNNING TRANSFORMER ARCHITECTURE TESTS ===")

test_model = TransformerModel(n_features=50, n_classes=7, d_model=64, n_heads=4, n_layers=2)
dummy_x = torch.randn(32, 10, 50) # Batch 32, Seq 10, Feat 50
dummy_y = torch.randint(0, 7, (32,))

# Test 1: Forward Pass Shape
logits = test_model(dummy_x)
assert logits.shape == (32, 7), "❌ FAILED Test 1: Output Shape Incorrect"
print("✅ PASSED Test 1: Forward Pass Output Shape is correct.")

# Test 2: Attention Weight Storage & Normalization
attn_weights = test_model.layers[0].attn.last_attention_weights
assert attn_weights is not None, "❌ FAILED Test 2: Attention weights not stored!"
assert np.allclose(attn_weights.sum(dim=-1).numpy(), 1.0, atol=1e-5), "❌ FAILED Test 2: Softmax failed"
print("✅ PASSED Test 2: Multi-Head Attention safely stores normalized weights (sum=1.0)!")

# Test 3: Gradient Flow (Pre-LN verification)
loss = test_model.criterion(logits, dummy_y)
loss.backward()
has_gradients = all(p.grad is not None for p in test_model.parameters() if p.requires_grad)
assert has_gradients, "❌ FAILED Test 3: Gradient vanishing detected!"
print("✅ PASSED Test 3: Backpropagation successfully traverses the entire Pre-LN network.")

# Test 4: Attention Rollout XAI
extractor = AttentionRolloutExtractor()
rollout_scores = extractor.extract(test_model, dummy_x)
assert rollout_scores.shape == (32, 10), "❌ FAILED Test 4: Rollout Shape"
assert np.allclose(rollout_scores.sum(axis=1), 1.0, atol=1e-5), "❌ FAILED Test 4: Rollout normalization"
print("✅ PASSED Test 4: Attention Rollout algorithm seamlessly extracts timeline importance.")

print("\n🎯 ALL TRANSFORMER ARCHITECTURE TESTS PASSED!")

=== RUNNING TRANSFORMER ARCHITECTURE TESTS ===


✅ PASSED Test 1: Forward Pass Output Shape is correct.
✅ PASSED Test 2: Multi-Head Attention safely stores normalized weights (sum=1.0)!


✅ PASSED Test 3: Backpropagation successfully traverses the entire Pre-LN network.
✅ PASSED Test 4: Attention Rollout algorithm seamlessly extracts timeline importance.

🎯 ALL TRANSFORMER ARCHITECTURE TESTS PASSED!
